# Install AI Access Manager

This notebook installs the **AI Access Manager** Streamlit app from this workspace into your Snowflake account. It:

1. Creates the target database and schema (`CREATE ... IF NOT EXISTS`)
2. Creates a stage and copies the app source files from this workspace into it
3. Creates the Streamlit app on the container runtime with access to Snowflake's shared PyPI artifact repository, and publishes it

**Run this notebook as `ACCOUNTADMIN`.** The app reads and changes account-wide grants, privileges, and parameters, so it must be owned by `ACCOUNTADMIN`.

Re-running is safe: an existing database, schema, and stage are reused, and the app is replaced with the current code from the workspace.

> **Disclaimer:** This is a community project and is **not** an official Snowflake product or offering. It is not supported, endorsed, or maintained by Snowflake Inc. The software is provided "as is", without warranty of any kind. Review the app's generated SQL carefully and test in a non-production account before use. Use at your own risk.

## 1. Configure

Edit the values below, then run all cells. Everything else in the notebook is driven by these variables.

In [ ]:
import os
import re

# ---- Edit these values before running ----
QUERY_WAREHOUSE = "COMPUTE_WH"               # warehouse for the app's SQL queries
APP_DATABASE    = "AI_ACCESS_MANAGER"        # created if it doesn't exist
APP_SCHEMA      = "AI_ACCESS_MANAGER_APP"    # created if it doesn't exist
APP_NAME        = "AI_ACCESS_MANAGER"        # Streamlit object name
APP_STAGE       = "APP_SOURCE"               # stage that holds a copy of the app source files
COMPUTE_POOL    = "SYSTEM_COMPUTE_POOL_CPU"  # Snowflake-provided default CPU compute pool

# Where the app source lives. Leave as None to auto-detect from the workspace this notebook runs in.
WORKSPACE_FQN   = None   # e.g. 'USER$.PUBLIC."ai-access-manager"'
APP_FOLDER      = None   # e.g. 'ai-access-manager' (folder containing streamlit_app.py, relative to the workspace root)
# -------------------------------------------

# Notebooks in workspaces expose the fully qualified workspace name, and the kernel's
# working directory is the folder this notebook lives in (the workspace root).
if WORKSPACE_FQN is None:
    WORKSPACE_FQN = os.environ.get("SNOWFLAKE_WORKSPACE_NAME")
    if not WORKSPACE_FQN:
        raise RuntimeError("Could not detect the workspace name; set WORKSPACE_FQN manually.")

if APP_FOLDER is None:
    _candidates = sorted(
        d for d in os.listdir(".")
        if os.path.isfile(os.path.join(d, "streamlit_app.py")) and os.path.isfile(os.path.join(d, "snowflake.yml"))
    )
    if len(_candidates) != 1:
        raise RuntimeError(f"Expected exactly one app folder next to this notebook, found {_candidates}; set APP_FOLDER manually.")
    APP_FOLDER = _candidates[0]

# These values are templated into SQL, so only allow plain identifiers / folder names.
_IDENT = re.compile(r"^[A-Za-z_][A-Za-z0-9_$]*$")
_PATH = re.compile(r"^[A-Za-z0-9_ -]+(/[A-Za-z0-9_ -]+)*$")
_FQN_PART = r'(?:[A-Za-z_][A-Za-z0-9_$]*|"[A-Za-z0-9_$ -]+")'
_FQN = re.compile(rf"^{_FQN_PART}\.{_FQN_PART}\.{_FQN_PART}$")

for _name in ["QUERY_WAREHOUSE", "APP_DATABASE", "APP_SCHEMA", "APP_NAME", "APP_STAGE", "COMPUTE_POOL"]:
    if not _IDENT.match(globals()[_name]):
        raise ValueError(f"{_name}={globals()[_name]!r} is not a valid unquoted Snowflake identifier")
if not _PATH.match(APP_FOLDER):
    raise ValueError(f"APP_FOLDER={APP_FOLDER!r} may only contain letters, digits, '_', '-', spaces and '/'")
if not _FQN.match(WORKSPACE_FQN):
    raise ValueError(f"WORKSPACE_FQN={WORKSPACE_FQN!r} is not a supported workspace name (expected DB.SCHEMA.\"name\")")

APP_FQN = f"{APP_DATABASE}.{APP_SCHEMA}.{APP_NAME}"
STAGE_FQN = f"{APP_DATABASE}.{APP_SCHEMA}.{APP_STAGE}"
SOURCE_URI = f"snow://workspace/{WORKSPACE_FQN}/versions/live/{APP_FOLDER}/"

print(f"App:     {APP_FQN}")
print(f"Stage:   @{STAGE_FQN}")
print(f"Source:  {SOURCE_URI}")
print(f"Compute: pool={COMPUTE_POOL}, warehouse={QUERY_WAREHOUSE}")

## 2. Create database, schema, and stage

In [ ]:
%%sql -r set_role
USE ROLE ACCOUNTADMIN

In [ ]:
%%sql -r create_database
CREATE DATABASE IF NOT EXISTS {{APP_DATABASE}}
  COMMENT = 'AI Access Manager (community project)'

In [ ]:
CREATE SCHEMA IF NOT EXISTS {{APP_DATABASE}}.{{APP_SCHEMA}}
  COMMENT = 'AI Access Manager Streamlit app and source files (community project)'

In [ ]:
%%sql -r create_stage
CREATE STAGE IF NOT EXISTS {{STAGE_FQN}}
  DIRECTORY = (ENABLE = TRUE)
  ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE')
  COMMENT = 'Source files for the AI Access Manager Streamlit app'

## 3. Copy the app files from the workspace

The stage is cleared first so files deleted from the workspace don't linger. Only `.py` and `.toml` files are copied, matching the `artifacts` in `snowflake.yml` (`uv.lock` and `snowflake.yml` are skipped).

In [ ]:
%%sql -r clear_stage
REMOVE @{{STAGE_FQN}}

In [ ]:
%%sql -r copy_files
COPY FILES INTO @{{STAGE_FQN}}
  FROM '{{SOURCE_URI}}'
  PATTERN = '.*[.](py|toml)$'

In [ ]:
%%sql -r staged_files
LIST @{{STAGE_FQN}}

## 4. Create and publish the Streamlit app

The app is attached to Snowflake's shared PyPI artifact repository (`snowflake.snowpark.pypi_shared_repository`) so it can install the dependencies in `pyproject.toml` (e.g. `streamlit[snowflake]>=1.65.0`). The owning role needs the `SNOWFLAKE.PYPI_REPOSITORY_USER` database role to use it.

In [ ]:
%%sql -r grant_pypi
-- Lets the app owner install packages from Snowflake's shared PyPI mirror.
-- No-op if ACCOUNTADMIN already has it.
GRANT DATABASE ROLE SNOWFLAKE.PYPI_REPOSITORY_USER TO ROLE ACCOUNTADMIN

In [ ]:
CREATE OR REPLACE STREAMLIT {{APP_FQN}}
  FROM '@{{STAGE_FQN}}'
  MAIN_FILE = 'streamlit_app.py'
  RUNTIME_NAME = 'SYSTEM$ST_CONTAINER_RUNTIME_PY3_11'
  COMPUTE_POOL = {{COMPUTE_POOL}}
  QUERY_WAREHOUSE = {{QUERY_WAREHOUSE}}
  ARTIFACT_REPOSITORIES = (snowflake.snowpark.pypi_shared_repository)
  TITLE = 'AI Access Manager'
  COMMENT = 'AI Access Manager (community project, not an official Snowflake offering)'

In [ ]:
%%sql -r publish_app
ALTER STREAMLIT {{APP_FQN}} ADD LIVE VERSION FROM LAST

In [ ]:
%%sql -r app_info
SHOW STREAMLITS LIKE '{{APP_NAME}}' IN SCHEMA {{APP_DATABASE}}.{{APP_SCHEMA}}

## 5. Next steps

- **Open the app:** in Snowsight go to **Projects » Streamlit** and select **AI Access Manager**. The first start can take a minute or two while the compute pool spins up the container and installs packages from the PyPI repository.
- **Update the app:** pull or edit the code in this workspace, then re-run this notebook.
- **Packages:** dependencies in `pyproject.toml` are installed only from `snowflake.snowpark.pypi_shared_repository`. Check the `artifact_repositories` column in the `SHOW STREAMLITS` output above to confirm it's attached.

> **Security warning — be careful who you share this app with.** Streamlit in Snowflake apps run with **owner's rights**, so this app acts as `ACCOUNTADMIN`. Granting `USAGE` on the app to another role effectively lets that role make `ACCOUNTADMIN`-level grant, privilege, and parameter changes through the app. This notebook does not grant access to any other role.